# Google Trends Explorer

Created by [Matt Artz](https://www.mattartz.me/) — Advancing AI Anthropology through computational approaches to qualitative research.

<br>

---

<br>

## What This Notebook Does

This notebook retrieves and visualizes Google Trends data for research purposes. Enter search terms, select a geography and time period, and the notebook returns interest-over-time data, regional breakdowns, and related queries — all exportable as structured data files for further analysis.

Google Trends measures relative search interest on a 0–100 scale, where 100 represents peak popularity for a term in the selected time period and geography. This data can help researchers understand public discourse, track emerging topics, and contextualize fieldwork within broader patterns of attention and interest.

## Key Features

1. **Multi-Term Comparison**: Compare up to five search terms simultaneously
2. **Geographic Filtering**: Query worldwide, by country, or by subregion (e.g., US-CA)
3. **Flexible Time Periods**: From the past hour to full history (2004–present)
4. **Regional Breakdown**: See which regions show the highest interest for your terms
5. **Related Queries**: Discover what else people search for alongside your terms
6. **Structured Export**: Download results as CSV, Excel, or JSON for further analysis

## Workflow

1. **Configure**: Set your search terms, geography, and time period
2. **Retrieve**: Fetch trend data from Google Trends
3. **Visualize**: Review interest-over-time charts and regional breakdowns
4. **Export**: Download structured data files for use in other tools

## Applications

This notebook supports research that benefits from understanding public search behavior — exploring how interest in topics shifts over time and across regions, identifying related concepts and search patterns, and contextualizing qualitative findings within broader trends in public attention. Useful for dissertation research, applied projects, and any study where understanding public discourse matters.

## Methodological Positioning

This notebook represents a **computational anthropology** approach — using structured data retrieval to complement traditional qualitative methods. Google Trends data reflects search interest, not absolute search volume, and should be interpreted comparatively rather than as a direct measure of public opinion or behavior.

**Important**: This notebook retrieves data but does not interpret it. Interpretation remains the work of the researcher.

## Target Audience

Designed for anthropologists and qualitative researchers who want to explore public search trends as part of their research process — from graduate students scoping dissertation topics to applied researchers contextualizing fieldwork findings.

## Technical Approach

The notebook uses the **pytrends** library (a community-maintained Python interface for Google Trends) to retrieve search interest data. All processing runs in the notebook — no external API keys required. Results are returned as pandas DataFrames for analysis and export.

## Contributing to AI Anthropology

This notebook contributes to the emerging field of AI Anthropology — which combines studying AI as cultural artifact, using AI to enhance ethnographic research, and applying anthropological insights to AI development (Artz, forthcoming). By open-sourcing these tools, this work advances the collective capacity of anthropologists to work effectively with computational methods.

## AI Anthropology Toolkit

This notebook is part of a growing suite of computational resources for anthropological research:

- **[Qualitative Codebook Builder](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit)**: AI-assisted development of qualitative coding frameworks with theory-driven code generation, inclusion/exclusion criteria, and export for analysis software
- **[Interview Transcript Semantic Chunker](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit)**: Segments interview transcripts into semantically coherent chunks with speaker-aware processing and multi-format support
- **[Coding and Thematic Analysis](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit)**: Applies codes to qualitative data and builds themes using deductive, inductive, or hybrid approaches with professional export

## License & Attribution

This notebook is released under the **CC BY-NC 4.0** license. You may adapt and share it for non-commercial purposes with proper attribution.

## Citation

> Artz, Matt. 2025. AI Anthropology Toolkit. Software. Zenodo. https://doi.org/10.5281/zenodo.16728812

## Setup and Installation

Install required Python packages and import libraries for Google Trends data retrieval, visualization, and interactive configuration. Run this cell first to ensure all dependencies are available.

In [ ]:
# Install required packages
!pip install pytrends pandas matplotlib openpyxl ipywidgets -q

import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import json
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

try:
    from pytrends.request import TrendReq
    HAVE_PYTRENDS = True
    print("\u2713 pytrends loaded successfully")
except ImportError:
    HAVE_PYTRENDS = False
    print("\u26a0\ufe0f pytrends not available \u2014 install with: pip install pytrends")

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def parse_keywords(raw):
    """Parse comma-separated keywords, max 5."""
    terms = [k.strip() for k in raw.split(",") if k.strip()]
    if len(terms) > 5:
        print(f"\u26a0\ufe0f Google Trends supports max 5 terms. Using first 5: {terms[:5]}")
        terms = terms[:5]
    return terms

print("\u2713 All packages installed and libraries loaded successfully")
print("\U0001f4cb Ready to configure your Google Trends query!")

## Query Configuration

Configure your Google Trends query using the interactive controls below. Set your search terms, geography, time period, and export preferences.

In [ ]:
# Configuration and Interactive Interface

class TrendsConfig:
    """Configuration for Google Trends queries."""
    KEYWORDS = "anthropology, ethnography"
    GEO = ""
    TIMEFRAME = "today 12-m"
    CATEGORY = 0
    OUTPUT_FORMAT = "excel"

config = TrendsConfig()

def create_configuration_interface():
    """Create branded configuration interface."""

    style = {'description_width': '140px'}
    layout = widgets.Layout(width='500px')

    instructions_html = '<div style="background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;">'
    instructions_html += '<h2 style="color: #274C77; margin-top: 0;">\U0001f50d Google Trends Explorer</h2>'
    instructions_html += '<p><strong>Welcome!</strong> This notebook retrieves and visualizes Google Trends data for research purposes.</p>'
    instructions_html += '<h3 style="color: #274C77;">\U0001f4d6 How to Use:</h3>'
    instructions_html += '<ol>'
    instructions_html += '<li><strong>Configure:</strong> Set your search terms and filters below</li>'
    instructions_html += '<li><strong>Retrieve:</strong> Run the next cell to fetch trend data</li>'
    instructions_html += '<li><strong>Visualize:</strong> Review charts and regional breakdowns</li>'
    instructions_html += '<li><strong>Export:</strong> Download structured data for further analysis</li>'
    instructions_html += '</ol>'
    instructions_html += '<p style="margin-top: 10px; color: #8B8C89; font-size: 0.9em;">'
    instructions_html += 'Google Trends data reflects relative search interest (0\u2013100 scale), not absolute search volume.</p>'
    instructions_html += '</div>'

    instructions = widgets.HTML(value=instructions_html)

    search_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f524 Search Terms</h3>')

    keywords_input = widgets.Textarea(
        value=config.KEYWORDS,
        placeholder='Enter up to 5 comma-separated terms',
        description='Keywords:',
        layout=widgets.Layout(width='500px', height='60px'),
        style=style
    )

    keywords_help = widgets.HTML('<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
        'Enter up to 5 terms separated by commas. Google Trends compares relative search interest across terms.</p>')

    geo_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f30d Geography & Time</h3>')

    geo_input = widgets.Text(
        value=config.GEO,
        placeholder='Leave empty for worldwide, or enter code (e.g., US, GB, US-CA)',
        description='Geography:',
        layout=layout,
        style=style
    )

    geo_help = widgets.HTML('<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
        'Use ISO 3166-1 alpha-2 for countries (US, GB, DE) or ISO 3166-2 for subregions (US-CA, GB-ENG). '
        'Leave empty for worldwide.</p>')

    timeframe_input = widgets.Dropdown(
        options=[
            ('Past hour', 'now 1-H'),
            ('Past 4 hours', 'now 4-H'),
            ('Past day', 'now 1-d'),
            ('Past 7 days', 'now 7-d'),
            ('Past 30 days', 'today 1-m'),
            ('Past 90 days', 'today 3-m'),
            ('Past 12 months', 'today 12-m'),
            ('Past 5 years', 'today 5-y'),
            ('2004\u2013present', 'all'),
        ],
        value=config.TIMEFRAME,
        description='Time period:',
        layout=layout,
        style=style
    )

    category_input = widgets.Dropdown(
        options=[
            ('All categories', 0),
            ('Arts & Entertainment', 3),
            ('Books & Literature', 22),
            ('Business & Industrial', 12),
            ('Computers & Electronics', 5),
            ('Health', 45),
            ('Jobs & Education', 958),
            ('Law & Government', 19),
            ('News', 16),
            ('People & Society', 14),
            ('Science', 174),
        ],
        value=config.CATEGORY,
        description='Category:',
        layout=layout,
        style=style
    )

    output_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f4c1 Output</h3>')

    output_format = widgets.Dropdown(
        options=[
            ('Excel (.xlsx)', 'excel'),
            ('CSV (.csv)', 'csv'),
            ('JSON (.json)', 'json'),
        ],
        value='excel',
        description='Export format:',
        layout=layout,
        style=style
    )

    save_btn = widgets.Button(
        description='Save Configuration',
        button_style='',
        layout=widgets.Layout(width='200px', margin='20px 0'),
        style={'button_color': '#6096BA'}
    )

    status = widgets.Output()

    def save_config(btn):
        config.KEYWORDS = keywords_input.value
        config.GEO = geo_input.value.strip()
        config.TIMEFRAME = timeframe_input.value
        config.CATEGORY = category_input.value
        config.OUTPUT_FORMAT = output_format.value
        with status:
            clear_output()
            print("\u2713 Configuration saved! Proceed to the next cell to retrieve data.")

    save_btn.on_click(save_config)

    display(widgets.VBox([
        instructions,
        search_header,
        keywords_input,
        keywords_help,
        geo_header,
        geo_input,
        geo_help,
        timeframe_input,
        category_input,
        output_header,
        output_format,
        save_btn,
        status,
    ]))

create_configuration_interface()

## Data Retrieval

Fetch Google Trends data based on your configuration. This retrieves interest over time, interest by region, and related queries for your search terms.

In [ ]:
# Retrieve Google Trends Data

def fetch_trends(config):
    """Fetch all trends data based on current configuration."""
    if not HAVE_PYTRENDS:
        raise RuntimeError("pytrends is not installed. Run the Setup cell first.")

    terms = parse_keywords(config.KEYWORDS)
    if not terms:
        raise ValueError("Please enter at least one search term.")

    geo_label = config.GEO if config.GEO else "Worldwide"
    print("\U0001f50d Querying Google Trends...")
    print(f"   Terms: {', '.join(terms)}")
    print(f"   Geography: {geo_label}")
    print(f"   Time period: {config.TIMEFRAME}")
    print()

    pytrends = TrendReq(hl='en-US', tz=0)

    pytrends.build_payload(
        kw_list=terms,
        geo=config.GEO,
        timeframe=config.TIMEFRAME,
        cat=config.CATEGORY,
    )

    results = {}

    print("\U0001f4c8 Fetching interest over time...")
    try:
        iot = pytrends.interest_over_time()
        if not iot.empty and 'isPartial' in iot.columns:
            iot = iot.drop(columns=['isPartial'])
        results['interest_over_time'] = iot
        print(f"   \u2713 {len(iot)} data points retrieved")
    except Exception as e:
        print(f"   \u26a0\ufe0f Could not retrieve interest over time: {e}")
        results['interest_over_time'] = pd.DataFrame()

    print("\U0001f30d Fetching interest by region...")
    try:
        ibr = pytrends.interest_by_region(resolution='COUNTRY', inc_low_vol=True, inc_geo_code=True)
        ibr = ibr[ibr.select_dtypes(include='number').sum(axis=1) > 0]
        results['interest_by_region'] = ibr
        print(f"   \u2713 {len(ibr)} regions with data")
    except Exception as e:
        print(f"   \u26a0\ufe0f Could not retrieve regional data: {e}")
        results['interest_by_region'] = pd.DataFrame()

    print("\U0001f517 Fetching related queries...")
    try:
        rq = pytrends.related_queries()
        results['related_queries'] = rq
        print("   \u2713 Related queries retrieved")
    except Exception as e:
        print(f"   \u26a0\ufe0f Could not retrieve related queries: {e}")
        results['related_queries'] = {}

    print()
    print("\u2713 Data retrieval complete!")
    return results

# Run the query
results = fetch_trends(config)

# Preview
if not results['interest_over_time'].empty:
    print()
    print("\U0001f4ca Interest Over Time (first 10 rows):")
    display(results['interest_over_time'].head(10))

## Visualization

Visualize the retrieved trends data with interest-over-time charts and regional breakdowns. Related queries are displayed as tables below the charts.

In [ ]:
# Visualize Trends Data

COLORS = ['#274C77', '#6096BA', '#A3CEF1', '#8B8C89', '#4A7C59']

def plot_interest_over_time(df, terms):
    """Plot interest over time with branded styling."""
    if df.empty:
        print("No interest-over-time data to plot.")
        return

    fig, ax = plt.subplots(figsize=(12, 5))
    fig.patch.set_facecolor('#FFFFFF')
    ax.set_facecolor('#FAFAFA')

    for i, term in enumerate(terms):
        if term in df.columns:
            color = COLORS[i % len(COLORS)]
            ax.plot(df.index, df[term], label=term, color=color, linewidth=2)

    ax.set_title('Google Trends \u2014 Interest Over Time',
                 fontsize=14, color='#274C77', fontweight='bold', pad=15)
    ax.set_xlabel('Date', fontsize=11, color='#274C77')
    ax.set_ylabel('Search Interest (relative)', fontsize=11, color='#274C77')
    ax.legend(frameon=True, facecolor='white', edgecolor='#E7ECEF')
    ax.grid(True, alpha=0.3, color='#8B8C89')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#E7ECEF')
    ax.spines['bottom'].set_color('#E7ECEF')

    plt.tight_layout()
    plt.show()


def plot_top_regions(df, terms, top_n=15):
    """Plot top regions by interest with branded styling."""
    if df.empty:
        print("No regional data to plot.")
        return

    term = terms[0] if terms[0] in df.columns else df.columns[0]
    top = df.nlargest(top_n, term)

    fig, ax = plt.subplots(figsize=(12, 6))
    fig.patch.set_facecolor('#FFFFFF')
    ax.set_facecolor('#FAFAFA')

    ax.barh(range(len(top)), top[term], color='#6096BA', edgecolor='#274C77', linewidth=0.5)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(top.index, fontsize=10)
    ax.invert_yaxis()

    ax.set_title(f'Top {top_n} Regions \u2014 "{term}"',
                 fontsize=14, color='#274C77', fontweight='bold', pad=15)
    ax.set_xlabel('Search Interest (relative)', fontsize=11, color='#274C77')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#E7ECEF')
    ax.spines['bottom'].set_color('#E7ECEF')
    ax.grid(True, axis='x', alpha=0.3, color='#8B8C89')

    plt.tight_layout()
    plt.show()


# Generate plots
terms = parse_keywords(config.KEYWORDS)

plot_interest_over_time(results['interest_over_time'], terms)
plot_top_regions(results['interest_by_region'], terms)

# Display related queries
if results.get('related_queries'):
    print()
    for term in terms:
        if term in results['related_queries']:
            data = results['related_queries'][term]
            if data.get('top') is not None and not data['top'].empty:
                print(f"\U0001f517 Top related queries for \"{term}\":")
                display(data['top'].head(10))
                print()
            if data.get('rising') is not None and not data['rising'].empty:
                print(f"\U0001f4c8 Rising related queries for \"{term}\":")
                display(data['rising'].head(10))
                print()

## Export

Export your trends data as CSV, Excel, or JSON. Files are saved to the current directory and downloaded automatically in Google Colab.

In [ ]:
# Export Results

def export_results(results, config):
    """Export trends data in the configured format."""
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    terms_slug = '_'.join(parse_keywords(config.KEYWORDS)[:3]).replace(' ', '')[:30]
    base_name = f'google_trends_{terms_slug}_{timestamp}'

    exported_files = []
    fmt = config.OUTPUT_FORMAT

    iot = results.get('interest_over_time', pd.DataFrame())
    ibr = results.get('interest_by_region', pd.DataFrame())

    if iot.empty and ibr.empty:
        print("\u26a0\ufe0f No data to export.")
        return

    if fmt == 'excel':
        filepath = f'{base_name}.xlsx'
        with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
            if not iot.empty:
                iot.to_excel(writer, sheet_name='Interest Over Time')
            if not ibr.empty:
                ibr.to_excel(writer, sheet_name='Interest By Region')

            rq = results.get('related_queries', {})
            terms = parse_keywords(config.KEYWORDS)
            for term in terms:
                if term in rq:
                    if rq[term].get('top') is not None and not rq[term]['top'].empty:
                        sheet_name = f'Related_{term[:20]}'
                        rq[term]['top'].to_excel(writer, sheet_name=sheet_name, index=False)

        # Style the Excel output
        from openpyxl import load_workbook
        from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

        wb = load_workbook(filepath)
        header_fill = PatternFill(start_color='274C77', end_color='274C77', fill_type='solid')
        header_font = Font(bold=True, color='FFFFFF')
        header_alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        thin_border = Border(
            left=Side(style='thin', color='E7ECEF'),
            right=Side(style='thin', color='E7ECEF'),
            top=Side(style='thin', color='E7ECEF'),
            bottom=Side(style='thin', color='E7ECEF'),
        )

        for ws in wb.worksheets:
            for cell in ws[1]:
                cell.fill = header_fill
                cell.font = header_font
                cell.alignment = header_alignment
                cell.border = thin_border

            for col in ws.columns:
                max_length = max(len(str(cell.value or '')) for cell in col)
                ws.column_dimensions[col[0].column_letter].width = min(max(max_length + 2, 10), 60)

            ws.freeze_panes = 'A2'

            for row in ws.iter_rows(min_row=2):
                for cell in row:
                    cell.alignment = Alignment(vertical='top', wrap_text=True)
                    cell.border = thin_border

        wb.save(filepath)
        exported_files.append(filepath)
        print(f"\u2713 Excel: {filepath}")

    elif fmt == 'csv':
        if not iot.empty:
            fp = f'{base_name}_interest_over_time.csv'
            iot.to_csv(fp)
            exported_files.append(fp)
            print(f"\u2713 CSV: {fp}")
        if not ibr.empty:
            fp = f'{base_name}_interest_by_region.csv'
            ibr.to_csv(fp)
            exported_files.append(fp)
            print(f"\u2713 CSV: {fp}")

    elif fmt == 'json':
        filepath = f'{base_name}.json'
        output = {
            'metadata': {
                'keywords': parse_keywords(config.KEYWORDS),
                'geography': config.GEO if config.GEO else 'Worldwide',
                'timeframe': config.TIMEFRAME,
                'retrieved_at': datetime.now().isoformat(),
            },
            'interest_over_time': json.loads(iot.reset_index().to_json(orient='records', date_format='iso')) if not iot.empty else [],
            'interest_by_region': json.loads(ibr.reset_index().to_json(orient='records')) if not ibr.empty else [],
        }
        with open(filepath, 'w') as f:
            json.dump(output, f, indent=2, default=str)
        exported_files.append(filepath)
        print(f"\u2713 JSON: {filepath}")

    # Auto-download in Colab
    if IN_COLAB and exported_files:
        print()
        for fp in exported_files:
            colab_files.download(fp)
            print(f"\U0001f4e5 Downloaded: {fp}")

export_results(results, config)